# Chapter 6: Autoencoders

This notebook accompanies **Chapter 6** of the lecture notes.

> The napkin still sits on your desk. Last lecture you found the directions of largest variance and used them to compress the digits. Today the same idea gets a new frame: a tiny network with a bottleneck in the middle, an encoder on one side, a decoder on the other. From there, a single line of regularisation turns the autoencoder into a generative model.

**Agenda**

🪞 · 📐 · 🎲 · 🆚 · 🎚️ · 🏁

**Take it from here:** 📊 · 🚶 · 🌡️

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_linear_autoencode, check_kl_divergence, check_reparameterize,
)

## 🪞 The Linear Autoencoder

An autoencoder is a network with a narrow waist: an encoder compresses each input into a low-dimensional code, a decoder reconstructs the input from that code, and the loss measures the round-trip error. Strip both networks down to a single matrix multiplication and the picture becomes very familiar.

> If the encoder is just multiplication by a matrix V.T, and the decoder is just multiplication by V, what does the loss `||x - V V.T x||^2` recover when V is chosen to minimise it?

<details><summary>Thought</summary>

It recovers PCA. Minimising the reconstruction error of a linear autoencoder over centred data picks V whose rows span the top-k principal directions of the covariance matrix. So the autoencoder you already built last lecture lives here, just renamed: encoder, decoder, bottleneck, loss.
</details>

We reuse the MNIST subset and the PCA components from lecture 05. The next cell loads the data, centres it, and computes V_k via numpy. Then you implement the forward pass.

In [ ]:
df = pd.read_csv('../lecture-05/mnist_digits.csv')
y  = df['label'].values
X  = df.drop('label', axis=1).values.astype(np.float64) / 255.0

# Centre and compute the top-k principal directions (numpy reference,
# so the rest of the notebook does not depend on lecture-05's pca_components).
X_c = X - X.mean(axis=0)
_cov = X_c.T @ X_c / (X_c.shape[0] - 1)
_e, _v = np.linalg.eigh(_cov)
_order = np.argsort(_e)[::-1]
V_full = _v[:, _order].T  # rows are components, sorted by descending eigenvalue

k = 20
V_k = V_full[:k]
print(f'X    : {X.shape}')
print(f'X_c  : {X_c.shape}')
print(f'V_k  : {V_k.shape}   (k = {k})')

In [ ]:
def linear_autoencode(X_c, V):
    """Run the linear autoencoder: encode to a k-dim latent, then decode.

    Returns
    -------
    Z       array of shape (n, k), the latent embeddings
    X_hat   array of shape (n, d), the reconstructions
    """
    # 1. Encode: project the centered data onto the principal directions.
    #    V has shape (k, d), so V.T has shape (d, k). Z = X_c @ V.T.
    # 2. Decode: map the latent codes back to the data space.
    #    Z has shape (n, k); multiplying by V (k, d) gives X_hat (n, d).
    # 3. Return both as a tuple (Z, X_hat).
    # YOUR CODE HERE
    pass


In [ ]:
check_linear_autoencode(linear_autoencode, X_c, V_k)

In [ ]:
_out = linear_autoencode(X_c, V_k)
if _out is None:
    print('⬜ Implement linear_autoencode above first.')
else:
    _Z, _X_hat = _out
    _X_hat_img = _X_hat + X.mean(axis=0)  # add the mean back for display

    _rng = np.random.default_rng(7)
    _idx = [int(_rng.choice(np.where(y == d)[0])) for d in range(10)]

    fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
    for col, i in enumerate(_idx):
        axes[0, col].imshow(X[i].reshape(28, 28), cmap='magma')
        axes[1, col].imshow(_X_hat_img[i].reshape(28, 28), cmap='magma')
        axes[0, col].set_title(f'{y[i]}', fontsize=9, color=_GOLDEN)
        for ax in (axes[0, col], axes[1, col]):
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
    axes[0, 0].set_ylabel('original', color=_TEXT)
    axes[1, 0].set_ylabel(f'k = {k}', color=_TEXT)
    plt.tight_layout()
    plt.show()

    # Reconstruction error vs k
    _ks = [2, 5, 10, 20, 50, 100, 200]
    _errs = []
    for _kk in _ks:
        _, _xh = linear_autoencode(X_c, V_full[:_kk])
        _errs.append(float(((X_c - _xh) ** 2).mean()))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(_ks, _errs, marker='o', color=_ACCENT, linewidth=1.2, markersize=6)
    ax.set_xlabel('Latent dimension k')
    ax.set_ylabel('Mean squared reconstruction error')
    ax.set_xscale('log')
    ax.set_title('Linear autoencoder, reconstruction vs bottleneck size', fontsize=10, color=_GOLDEN)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- The k = 20 reconstructions are recognisable digits, slightly blurred. The bottleneck has discarded high-frequency detail and kept the global stroke layout.
- The error curve drops fast at first, then flattens. Past a point, adding more components only buys diminishing returns.
- This is the same picture the lecture notes describe in words: the autoencoder is forced to choose what to keep. PCA is the optimal linear choice.

## 📐 The KL Divergence

The autoencoder you just built is a deterministic compressor. Every input maps to one fixed point in the latent space, and there is nothing to stop those points from spreading anywhere they like. The variational autoencoder breaks both habits: each input maps to a Gaussian, and a regulariser pulls those Gaussians toward a fixed prior.

> If the regulariser pulls every encoder distribution toward N(0, I), what stops the model from collapsing all inputs onto the same point?

<details><summary>Thought</summary>

The reconstruction term pushes back. If two inputs were encoded to the same point, the decoder could not tell them apart and the reconstruction loss would explode. Training finds an equilibrium: each input gets a slightly shifted, roughly unit-variance Gaussian, just spread out enough for the decoder to reconstruct.
</details>

The regulariser is a closed-form expression. For a diagonal-covariance Gaussian against the standard normal prior:

    KL = -0.5 * sum( 1 + log_var - mu**2 - exp(log_var) )

summed over the latent axis. Implement it now.

In [ ]:
def kl_divergence(mu, log_var):
    """Closed-form KL between N(mu, exp(log_var)) and N(0, I), per sample."""
    # The closed form for one sample with diagonal covariance:
    #   KL = -0.5 * sum_j ( 1 + log_var_j - mu_j^2 - exp(log_var_j) )
    # 1. Compute the variance: sigma_squared = np.exp(log_var).
    # 2. Combine the four terms inside the parentheses.
    # 3. Sum over the latent axis (axis=1) to get one KL per sample.
    # 4. Multiply by -0.5 and return.
    # YOUR CODE HERE
    pass


In [ ]:
_mu = np.array([[0.0, 0.0], [2.0, 0.0], [0.0, 0.0], [1.5, -1.5]])
_lv = np.array([[0.0, 0.0], [0.0, 0.0], [np.log(4.0), np.log(4.0)], [-1.0, -1.0]])
check_kl_divergence(kl_divergence, _mu, _lv)

In [ ]:
_kl_test = kl_divergence(np.zeros((1, 2)), np.zeros((1, 2)))
if _kl_test is None:
    print('⬜ Implement kl_divergence above first.')
else:
    _mus = np.linspace(-3, 3, 60)
    _sigs = np.linspace(0.1, 3.0, 60)
    _MU, _SIG = np.meshgrid(_mus, _sigs)
    _LV = np.log(_SIG ** 2)
    _KL = np.empty_like(_MU)
    for i in range(_MU.shape[0]):
        for j in range(_MU.shape[1]):
            _KL[i, j] = float(kl_divergence(
                np.array([[_MU[i, j]]]), np.array([[_LV[i, j]]])
            )[0])

    fig, ax = plt.subplots(figsize=(7, 5))
    _im = ax.pcolormesh(_MU, _SIG, _KL, cmap='magma', shading='auto')
    ax.contour(_MU, _SIG, _KL, levels=[0.1, 0.5, 1, 2, 4, 8], colors=_TEXT, linewidths=0.6)
    ax.scatter([0], [1], color=_ACCENT, s=60, zorder=3, label='prior (mu=0, sigma=1)')
    ax.set_xlabel('mu')
    ax.set_ylabel('sigma')
    ax.set_title('KL( N(mu, sigma^2) || N(0, 1) )', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, labelcolor=_TEXT, loc='upper right')
    fig.colorbar(_im, ax=ax, fraction=0.046, pad=0.04, label='KL')
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- The minimum sits exactly at (mu = 0, sigma = 1), where the encoder distribution equals the prior.
- The penalty grows quadratically as the mean drifts from zero, and asymmetrically with the variance: shrinking sigma below 1 costs more than enlarging it.
- Read in reverse: the KL term tells the encoder where the prior wants every distribution to land. The reconstruction term is what stops it from getting there.

## 🎲 The Reparameterisation Trick

Sampling is in the way of training. If z is drawn directly from N(mu, sigma^2), gradients cannot flow back through the random draw to update mu and sigma. The reparameterisation trick separates the deterministic and the stochastic parts, so the gradient signal only ever passes through deterministic operations.

> Why does pulling epsilon from N(0, I) and computing z = mu + sigma * eps make the network differentiable through z?

<details><summary>Thought</summary>

Because mu and sigma are deterministic functions of x and the network parameters; only eps is random, and eps does not depend on those parameters. The chain rule then has a clean path: dz/dmu = 1 and dz/dsigma = eps. The randomness sits outside the computational graph.
</details>

Implement `reparameterize(mu, log_var, rng)`. Use `log_var` instead of `sigma` for numerical stability — that is the usual convention and matches the KL formula above.

In [ ]:
def reparameterize(mu, log_var, rng):
    """Sample z = mu + sigma * eps with sigma = exp(0.5 * log_var)."""
    # 1. Convert log-variance to standard deviation:
    #    sigma = np.exp(0.5 * log_var). The 0.5 is essential — exp(log_var)
    #    is the variance, but we want the standard deviation.
    # 2. Draw epsilon from a standard normal, same shape as mu:
    #    eps = rng.standard_normal(mu.shape)
    # 3. Return mu + sigma * eps. Element-wise multiplication, not matmul.
    # YOUR CODE HERE
    pass


In [ ]:
_mu_test = np.array([[1.0, -2.0]])
_lv_test = np.array([[np.log(0.25), np.log(1.0)]])  # sigma = 0.5, 1.0
check_reparameterize(reparameterize, _mu_test, _lv_test, rng_seed=0)

In [ ]:
_mu_demo = np.array([[1.0, -2.0]])
_lv_demo = np.array([[np.log(0.25), np.log(1.0)]])
_rng = np.random.default_rng(0)
_samples = np.vstack([reparameterize(_mu_demo, _lv_demo, _rng) for _ in range(500)])

if _samples is None or _samples.shape != (500, 2):
    print('⬜ Implement reparameterize above first.')
else:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(_samples[:, 0], _samples[:, 1], s=10, color=_ACCENT, alpha=0.5, linewidths=0)
    ax.scatter([_mu_demo[0, 0]], [_mu_demo[0, 1]], s=80, color=_GOLDEN, zorder=3, label='mu')
    _theta = np.linspace(0, 2 * np.pi, 200)
    for _r in (1, 2):
        ax.plot(
            _mu_demo[0, 0] + _r * 0.5 * np.cos(_theta),
            _mu_demo[0, 1] + _r * 1.0 * np.sin(_theta),
            color=_TEXT, linewidth=0.6, linestyle='--', alpha=0.4,
        )
    ax.set_xlabel('z_1'); ax.set_ylabel('z_2')
    ax.set_title('500 samples of z = mu + sigma * eps', fontsize=10, color=_GOLDEN)
    ax.legend(frameon=False, labelcolor=_TEXT, loc='upper right')
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()

**Observe:**
- The cloud is anchored at mu and stretched along each axis by the corresponding sigma. Different sigmas in different dimensions produce an ellipse, not a circle.
- Repeated calls give different samples but the same statistics. mu and sigma are deterministic; only eps changes between draws.
- This is the gradient-friendly version of "draw from N(mu, sigma^2)". Backprop sees a fixed path through mu and sigma.

## 🆚 Embeddings With and Without the KL

The two visualisations below come from precomputed embeddings of the same MNIST subset, encoded by two networks trained side by side: a plain autoencoder and a variational autoencoder, both with k = 2. The only architectural difference is the KL term in the VAE's loss.

> Knowing what KL does to (mu, sigma) in isolation, what should change in the global layout when KL is applied to every input simultaneously?

<details><summary>Thought</summary>

Each input's distribution is pulled toward the same prior, so the cloud of mus contracts toward the origin and the per-dimension scale becomes uniform. The aggregate of all encoder distributions ends up looking like the prior. Without KL, the autoencoder is free to spread embeddings however reconstruction prefers, with no global organisation.
</details>

In [ ]:
_em = pd.read_csv('embeddings_k2.csv')
_cmap = plt.get_cmap('tab10', 10)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=False)
for _digit in range(10):
    _m = _em['label'] == _digit
    axes[0].scatter(_em.loc[_m, 'ae_z1'], _em.loc[_m, 'ae_z2'], color=_cmap(_digit),
                    s=10, linewidths=0, alpha=0.7, label=str(_digit))
    axes[1].scatter(_em.loc[_m, 'vae_mu1'], _em.loc[_m, 'vae_mu2'], color=_cmap(_digit),
                    s=10, linewidths=0, alpha=0.7, label=str(_digit))

_theta = np.linspace(0, 2 * np.pi, 200)
for _r in (1, 2):
    axes[1].plot(_r * np.cos(_theta), _r * np.sin(_theta),
                 color=_TEXT, linewidth=0.6, linestyle='--', alpha=0.4)

axes[0].set_title('AE  (no KL term)', fontsize=10, color=_GOLDEN)
axes[1].set_title('VAE  (KL toward N(0, I))', fontsize=10, color=_GOLDEN)
for ax in axes:
    ax.set_xlabel('z_1'); ax.set_ylabel('z_2')
    tufte_axis(ax)
axes[1].legend(ncol=5, frameon=False, fontsize=8, markerscale=1.6,
               labelcolor=_TEXT, bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

**Observe:**
- The AE plot spreads classes across a wide, irregular range. There are gaps between digit clusters; sampling from those gaps would give the decoder a code it has never been asked to decode.
- The VAE plot is contained inside the prior's natural scale (the dashed circles mark 1 and 2 sigma of the standard normal). Classes are still separated, but every region of the latent space is now near a training point.
- The KL term traded a little reconstruction fidelity for a latent space you can sample from. That is what makes the VAE generative.

## 🎚️ Bottleneck Size

The latent dimensionality k controls how much information the bottleneck can carry. Too small and the network cannot encode the structure of the data; too large and most dimensions become unused while reconstruction stops improving. The two loss terms respond differently to k - one keeps falling, the other saturates.

> If you train the same VAE at progressively larger k, which loss component should keep falling and which should plateau, and why?

<details><summary>Thought</summary>

Reconstruction keeps falling, with diminishing returns: each extra dimension carries a little less information than the one before. KL plateaus, because once the latent space has enough room to hold the data, extra dimensions collapse to the prior, pay no KL, and contribute no decoding signal. The two curves split: one is bounded from below by the data's information content, the other from above by exactly the same thing.
</details>

The cell below loads per-epoch reconstruction and KL traces for VAEs trained at k in {2, 4, 8, 16, 32}, all on the same 1000-sample MNIST subset, 80 epochs each.

In [ ]:
_curves = pd.read_csv('bottleneck_curves.csv')
_curve_ks = [2, 4, 8, 16, 32]
_curve_colors = plt.get_cmap('magma')(np.linspace(0.25, 0.85, len(_curve_ks)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for _kk, _c in zip(_curve_ks, _curve_colors):
    axes[0].plot(_curves['epoch'], _curves[f'k{_kk}_recon'],
                 color=_c, linewidth=1.2, label=f'k = {_kk}')
    axes[1].plot(_curves['epoch'], _curves[f'k{_kk}_kl'],
                 color=_c, linewidth=1.2, label=f'k = {_kk}')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Reconstruction loss')
axes[0].set_title('Reconstruction vs epoch', fontsize=10, color=_GOLDEN)
axes[0].legend(frameon=False, labelcolor=_TEXT, ncol=2, fontsize=9)
tufte_axis(axes[0])

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('KL divergence')
axes[1].set_title('KL vs epoch', fontsize=10, color=_GOLDEN)
axes[1].legend(frameon=False, labelcolor=_TEXT, ncol=2, fontsize=9)
tufte_axis(axes[1])

plt.tight_layout()
plt.show()

**Observe:**
- Reconstruction falls fastest at small k. Going from k = 2 to k = 8 buys most of the improvement; from 8 to 32 the curves stack on top of each other.
- KL rises with k but only up to a point. Past k = 8 it plateaus around the same value, regardless of how much extra capacity you hand the model. The dataset has a finite amount of information to encode and the extra dimensions collapse to the prior.
- The two panels are the two faces of the same trade-off. Reconstruction tells you "did the bottleneck have enough capacity?" KL tells you "how much of the capacity is the model actually using?"

### 🏁 Recap

**What we did:**
- 🪞 Reframed PCA as a linear autoencoder, and saw that the linear case has a closed-form optimum.
- 📐 Implemented the closed-form KL divergence between an encoder distribution and the standard normal prior, and saw where its minimum sits.
- 🎲 Implemented the reparameterisation trick that lets gradients flow through a stochastic sampling step.
- 🆚 Compared AE and VAE embeddings of MNIST. The KL term reorganises the latent space around the prior at the cost of a little reconstruction fidelity.
- 🎚️ Watched reconstruction quality and active capacity respond to the bottleneck size.

**Key takeaways:**
- An autoencoder is an unsupervised learner. The training signal is just the input itself.
- The KL term is the difference between an autoencoder and a generative model. It makes the latent space samplable.
- The reparameterisation trick is what makes the VAE trainable end to end. Everything else follows from that one line of algebra.
- Bottleneck size is the most consequential hyperparameter; the rest are tuning knobs around it.

## Take It from Here, Next Steps

These two sections are optional. They use precomputed assets from a slightly larger VAE (k = 16) to illustrate two diagnostic and exploratory ideas from the lecture notes.

### 📊 Active Units

When the latent space is large enough that the model has spare capacity, only some dimensions end up carrying signal. The rest collapse to the prior and pay no KL. The mean KL per dimension is a direct readout of which dimensions are doing work.

> If a dimension's mean KL is close to zero, what is the encoder doing for that dimension across the dataset?

<details><summary>Thought</summary>

It is outputting (mu = 0, sigma = 1) for every input — exactly the prior. The decoder gets no information from that dimension, and the dimension is effectively dead.
</details>

The cell below loads the (mu, log_var) of a k = 16 VAE and uses your `kl_divergence` to compute the per-dimension mean KL across the dataset.

In [ ]:
_au = pd.read_csv('active_units_k16.csv')

fig, ax = plt.subplots(figsize=(10, 4))
_colors = [_TERRA if v < 0.1 else _ACCENT for v in _au['mean_kl']]
ax.bar(_au['dim'], _au['mean_kl'], color=_colors, edgecolor=_BORDER, linewidth=0.6)
ax.axhline(0.1, color=_GOLDEN, linewidth=0.8, linestyle='--', label='active threshold (0.1)')
ax.set_xlabel('Latent dimension')
ax.set_ylabel('Mean KL across dataset')
ax.set_title('Active units in a k = 16 VAE', fontsize=10, color=_GOLDEN)
ax.legend(frameon=False, labelcolor=_TEXT)
tufte_axis(ax)
plt.tight_layout()
plt.show()

_n_active = int((_au['mean_kl'] > 0.1).sum())
print(f'Active dimensions: {_n_active} of {len(_au)}')

**Observe:**
- Some bars are near zero. Those dimensions have collapsed to the prior; the encoder ignores the input and outputs (0, 1) every time.
- The active dimensions correspond to factors of variation the model has chosen to encode. Their count is an estimate of how much information the data actually requires.
- This is a diagnostic for choosing k. If only six of sixteen dimensions are alive, you can drop k toward six without hurting reconstruction.

### 🚶 A Walk Through the Latent Space

Linear interpolation in the latent space, decoded image by image, traces a path through the manifold the VAE has learned. The example below comes from the same k = 16 VAE: pick a digit zero, pick a digit one, lerp between their encoded means, decode each waypoint.

Smooth interpolation is the simplest evidence that the encoder has learned a continuous feature space rather than a scatter of disconnected embeddings.

In [ ]:
_walk = pd.read_csv('latent_walk.csv')
_imgs = _walk.drop('t', axis=1).values
_ts   = _walk['t'].values

fig, axes = plt.subplots(1, len(_imgs), figsize=(14, 1.8))
for ax, img, t in zip(axes, _imgs, _ts):
    ax.imshow(img.reshape(28, 28), cmap='magma')
    ax.set_title(f't = {t:.1f}', fontsize=8, color=_GOLDEN)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout()
plt.show()

**Observe:**
- The endpoints look like a digit 0 and a digit 1, as expected.
- The intermediate frames are not training data; the decoder is reading out parts of the latent space that no specific input was ever encoded to. The fact that those readouts still look like digits is the geometric guarantee the KL term bought us.
- This is the simplest face of generative AI: a smooth latent space plus a trained decoder is, in essence, a generator. Modern image, text, and video models scale up the same trick.

### 🌡️ KL Annealing

The VAE loss combines two pulls: reconstruction wants spread, KL wants every encoder distribution close to the prior. With both at full strength from epoch zero, the KL term often wins early. Many dimensions collapse to the prior before the decoder has had time to learn anything useful, and reconstruction never recovers. KL annealing fixes this by starting beta at zero and ramping it up over the first few dozen epochs - the encoder spreads first, the regulariser tightens later. The effect is sharpest when beta is large enough that constant-from-zero training cannot recover.

> If you train with constant beta = 4 from epoch zero, what does the encoder do during the first few updates, and why is that hard to reverse?

<details><summary>Thought</summary>

The KL term immediately punishes any output that is not (mu = 0, log_var = 0). Reconstruction is large but its gradients are scattered across the whole network, while the KL gradient points cleanly at "make every dimension the prior". Most dimensions collapse before the decoder has any signal to defend them, and once collapsed they pay no KL and receive no useful gradient back. They stay dead.
</details>

The cell below loads per-epoch metrics from two k = 32 VAE training runs - one with constant beta = 4 from the first epoch, one with beta linearly warmed up from 0 to 4 over the first 40 of 150 epochs. Same seed, same architecture, same data.

In [ ]:
_an = pd.read_csv('kl_annealing.csv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(_an['epoch'], _an['beta_const'],  color=_GOLDEN, linewidth=1.2, label='constant')
axes[0].plot(_an['epoch'], _an['beta_anneal'], color=_ACCENT, linewidth=1.2, label='annealed')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('beta')
axes[0].set_title('beta schedule', fontsize=10, color=_GOLDEN)
axes[0].legend(frameon=False, labelcolor=_TEXT)
tufte_axis(axes[0])

axes[1].plot(_an['epoch'], _an['recon_const'],  color=_GOLDEN, linewidth=1.2, label='constant')
axes[1].plot(_an['epoch'], _an['recon_anneal'], color=_ACCENT, linewidth=1.2, label='annealed')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Reconstruction loss')
axes[1].set_title('Reconstruction', fontsize=10, color=_GOLDEN)
axes[1].legend(frameon=False, labelcolor=_TEXT)
tufte_axis(axes[1])

# KL goes log-scale so the early annealed-run spike (encoder spreading
# freely while beta is still near zero) and the late-game steady state
# are both legible on the same axis.
axes[2].plot(_an['epoch'], _an['kl_const'],  color=_GOLDEN, linewidth=1.2, label='constant')
axes[2].plot(_an['epoch'], _an['kl_anneal'], color=_ACCENT, linewidth=1.2, label='annealed')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('KL (log scale)')
axes[2].set_yscale('log')
axes[2].set_title('KL divergence', fontsize=10, color=_GOLDEN)
axes[2].legend(frameon=False, labelcolor=_TEXT)
tufte_axis(axes[2])

plt.tight_layout()
plt.show()

In [ ]:
_aa = pd.read_csv('kl_annealing_active.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, _col, _title in [
    (axes[0], 'mean_kl_const',  'constant beta = 4'),
    (axes[1], 'mean_kl_anneal', 'annealed 0 -> 4 over 40 epochs'),
]:
    _colors = [_TERRA if v < 0.1 else _ACCENT for v in _aa[_col]]
    ax.bar(_aa['dim'], _aa[_col], color=_colors, edgecolor=_BORDER, linewidth=0.6)
    ax.axhline(0.1, color=_GOLDEN, linewidth=0.8, linestyle='--')
    ax.set_xlabel('Latent dimension')
    ax.set_title(_title, fontsize=10, color=_GOLDEN)
    tufte_axis(ax)
axes[0].set_ylabel('Mean KL across dataset')
plt.tight_layout()
plt.show()

_n_const  = int((_aa['mean_kl_const']  > 0.1).sum())
_n_anneal = int((_aa['mean_kl_anneal'] > 0.1).sum())
print(f'Active dimensions  - constant: {_n_const}/{len(_aa)},  annealed: {_n_anneal}/{len(_aa)}')

**Observe:**
- The constant-beta KL collapses to near zero within the first few epochs and never escapes. With beta = 4 hitting from cold start, every dimension gets shoved onto the prior before the decoder has any signal to defend them.
- The annealed KL is allowed to rise while beta is small, peaks somewhere mid-warmup, then settles into a steady regime as beta tightens. The trajectory has the characteristic "rise and fall" shape of a successful warmup.
- Reconstruction tracks the same story in reverse: the constant run plateaus at a higher floor, the annealed run keeps falling and settles lower. The gap is small in absolute terms but stable - those few extra nats of KL are paying for real reconstruction.
- The bar chart below makes the consequence concrete. The annealed run keeps more dimensions alive, and the surviving dims carry noticeably more information per dim than anything in the constant run.
- Annealing is a training trick, not a model change. The forward pass and the loss are exactly the same VAE - only the weight on the KL term varies with the epoch.